# U.S. Flight Delays — Tableau Dashboard
**Summer 2025** | Tools: Python, Pandas, Matplotlib, Seaborn, Tableau Public

Analysis of 1M+ real U.S. domestic flights from the Bureau of Transportation Statistics (BTS).  
This notebook cleans the raw data, engineers delay metrics, and exports a Tableau-ready CSV used to build an interactive dashboard analyzing cancellations, ground delays, weather impact, and airport-level congestion patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 115
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('data/flight_data_2024.csv', parse_dates=['fl_date'])
print(f"Rows loaded : {len(df):,}")
print(f"Columns     : {df.columns.tolist()}")
print(f"Date range  : {df['fl_date'].min().date()} → {df['fl_date'].max().date()}")
df.head()

## 1. Data Cleaning & Feature Engineering

In [ ]:
# Day of week labels
dow_map = {1:'Monday', 2:'Tuesday', 3:'Wednesday', 4:'Thursday',
           5:'Friday', 6:'Saturday', 7:'Sunday'}
month_map = {1:'January', 2:'February', 3:'March', 4:'April',
             5:'May', 6:'June', 7:'July', 8:'August',
             9:'September', 10:'October', 11:'November', 12:'December'}

df['day_name']   = df['day_of_week'].map(dow_map)
df['month_name'] = df['month'].map(month_map)

# Total delay per flight (weather + late aircraft — the two delay cause columns available)
df['total_delay'] = df['weather_delay'] + df['late_aircraft_delay']

# Flag: flight experienced meaningful delay (taxi out > 30 min is a congestion signal)
df['high_taxi_out']  = (df['taxi_out'] > 30).astype(int)
df['is_delayed']     = (df['total_delay'] > 15).astype(int)
df['delay_category'] = pd.cut(
    df['total_delay'],
    bins=[-1, 0, 15, 45, 120, df['total_delay'].max()+1],
    labels=['None', 'Minor (1–15 min)', 'Moderate (16–45 min)',
            'Significant (46–120 min)', 'Severe (120+ min)']
)

# Primary delay cause
def primary_cause(row):
    if row['cancelled'] == 1: return 'Cancelled'
    if row['weather_delay'] > row['late_aircraft_delay'] and row['weather_delay'] > 0:
        return 'Weather'
    if row['late_aircraft_delay'] > 0: return 'Late Aircraft'
    if row['high_taxi_out']:           return 'Ground Congestion'
    return 'On Time'

df['primary_cause'] = df.apply(primary_cause, axis=1)

print(f"Flights with any delay     : {df['is_delayed'].sum():,} ({df['is_delayed'].mean()*100:.1f}%)")
print(f"Cancelled                  : {df['cancelled'].sum():,} ({df['cancelled'].mean()*100:.1f}%)")
print(f"High taxi-out (>30 min)    : {df['high_taxi_out'].sum():,} ({df['high_taxi_out'].mean()*100:.1f}%)")
print(f"Weather delays             : {(df['weather_delay']>0).sum():,}")
print(f"Late aircraft delays       : {(df['late_aircraft_delay']>0).sum():,}")
df[['total_delay','delay_category','primary_cause','is_delayed','high_taxi_out']].head(8)

## 2. Dashboard KPIs

In [ ]:
total      = len(df)
cancelled  = df['cancelled'].sum()
delayed    = df['is_delayed'].sum()
avg_weather= df[df['weather_delay']>0]['weather_delay'].mean()
avg_late   = df[df['late_aircraft_delay']>0]['late_aircraft_delay'].mean()
avg_taxi   = df['taxi_out'].mean()

print("=" * 40)
print("       DASHBOARD KPIs")
print("=" * 40)
print(f"  Total Flights          : {total:>10,}")
print(f"  Cancelled              : {cancelled:>10,}  ({cancelled/total*100:.1f}%)")
print(f"  Delayed (>15 min)      : {delayed:>10,}  ({delayed/total*100:.1f}%)")
print(f"  Avg Weather Delay      : {avg_weather:>10.1f} min")
print(f"  Avg Late Aircraft Delay: {avg_late:>10.1f} min")
print(f"  Avg Taxi-Out Time      : {avg_taxi:>10.1f} min")
print("=" * 40)

## 3. Cancellations by Airport

In [ ]:
airport_stats = df.groupby('origin').agg(
    total_flights   = ('cancelled', 'count'),
    cancellations   = ('cancelled', 'sum'),
    weather_delay   = ('weather_delay', 'mean'),
    late_aircraft   = ('late_aircraft_delay', 'mean'),
    avg_taxi_out    = ('taxi_out', 'mean'),
    high_taxi_flights = ('high_taxi_out', 'sum'),
).reset_index()

airport_stats['cancel_rate']   = (airport_stats['cancellations'] / airport_stats['total_flights'] * 100).round(2)
airport_stats['high_taxi_rate'] = (airport_stats['high_taxi_flights'] / airport_stats['total_flights'] * 100).round(2)

top_cancel = airport_stats[airport_stats['total_flights'] > 3000].sort_values('cancel_rate', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#c0392b' if r > top_cancel['cancel_rate'].mean() else '#2980b9'
          for r in top_cancel['cancel_rate']]
bars = ax.barh(top_cancel['origin'], top_cancel['cancel_rate'], color=colors)
ax.axvline(top_cancel['cancel_rate'].mean(), color='gray', linestyle='--', linewidth=1.2, label='Average')
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontsize=9)
ax.set_title('Cancellation Rate by Airport (Top 15, min 3K flights)', fontweight='bold')
ax.set_xlabel('Cancellation Rate (%)')
ax.set_ylabel('Airport Code')
ax.legend()
plt.tight_layout()
plt.savefig('cancellation_by_airport.png', bbox_inches='tight')
plt.show()

## 4. Ground Congestion — Taxi-Out Time by Airport

In [ ]:
top_taxi = airport_stats[airport_stats['total_flights'] > 3000].sort_values('avg_taxi_out', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette('Blues_r', n_colors=len(top_taxi))
bars = ax.barh(top_taxi['origin'], top_taxi['avg_taxi_out'], color=palette)
ax.axvline(df['taxi_out'].mean(), color='crimson', linestyle='--', linewidth=1.2,
           label=f'National Avg ({df["taxi_out"].mean():.1f} min)')
for bar in bars:
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f} min', va='center', fontsize=9)
ax.set_title('Average Taxi-Out Time by Airport (Ground Congestion)', fontweight='bold')
ax.set_xlabel('Avg Taxi-Out Time (minutes)')
ax.set_ylabel('Airport Code')
ax.legend()
plt.tight_layout()
plt.savefig('taxi_out_by_airport.png', bbox_inches='tight')
plt.show()

## 5. Delay Causes Breakdown

In [ ]:
cause_counts = df['primary_cause'].value_counts()
cause_pct    = (cause_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2ecc71','#3498db','#e67e22','#c0392b','#8e44ad']

axes[0].pie(cause_counts.values, labels=cause_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize':10})
axes[0].set_title('Primary Delay / Cancellation Cause', fontweight='bold')

delay_only = df[df['total_delay'] > 0]
axes[1].hist(delay_only['total_delay'].clip(upper=180), bins=40,
             color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of Delay Duration (flights with delay > 0)', fontweight='bold')
axes[1].set_xlabel('Total Delay (minutes, capped at 180)')
axes[1].set_ylabel('Number of Flights')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)} min'))

plt.tight_layout()
plt.savefig('delay_causes.png', bbox_inches='tight')
plt.show()

## 6. Day of Week Patterns

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow_stats = df.groupby('day_name').agg(
    total          = ('cancelled','count'),
    cancellations  = ('cancelled','sum'),
    avg_taxi       = ('taxi_out','mean'),
    avg_delay      = ('total_delay','mean')
).reindex(dow_order).reset_index()

dow_stats['cancel_rate'] = (dow_stats['cancellations'] / dow_stats['total'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = sns.color_palette('Blues', n_colors=7)
sns.barplot(data=dow_stats, x='day_name', y='cancel_rate', palette=palette, ax=axes[0])
axes[0].set_title('Cancellation Rate by Day of Week', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Cancellation Rate (%)')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=dow_stats, x='day_name', y='avg_taxi', palette=palette, ax=axes[1])
axes[1].set_title('Average Taxi-Out Time by Day of Week', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Avg Taxi-Out (min)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(df['taxi_out'].mean(), color='crimson', linestyle='--', linewidth=1.2, label='Overall Avg')
axes[1].legend()

plt.tight_layout()
plt.savefig('day_of_week.png', bbox_inches='tight')
plt.show()

## 7. State-Level Heatmap (for Tableau Map)

In [ ]:
state_stats = df.groupby('origin_state_nm').agg(
    total_flights  = ('cancelled','count'),
    cancellations  = ('cancelled','sum'),
    avg_taxi_out   = ('taxi_out','mean'),
    avg_weather    = ('weather_delay','mean'),
    avg_late       = ('late_aircraft_delay','mean'),
).reset_index()

state_stats['cancel_rate']      = (state_stats['cancellations'] / state_stats['total_flights'] * 100).round(2)
state_stats['avg_total_delay']  = (state_stats['avg_weather'] + state_stats['avg_late']).round(2)
state_stats = state_stats.sort_values('total_flights', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top_states = state_stats.head(20)
palette = sns.color_palette('Blues_r', n_colors=len(top_states))
bars = ax.barh(top_states['origin_state_nm'], top_states['total_flights'], color=palette)
ax.set_title('Total Flights by State (Top 20)', fontweight='bold')
ax.set_xlabel('Number of Flights')
ax.set_ylabel('')
for bar in bars:
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('flights_by_state.png', bbox_inches='tight')
plt.show()
state_stats.head(10)

## 8. Distance vs. Air Time Efficiency

In [ ]:
sample = df[df['cancelled'] == 0].sample(15000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(sample['distance'], sample['air_time'],
                     c=sample['taxi_out'], cmap='RdYlGn_r',
                     alpha=0.3, s=8, vmin=5, vmax=45)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Taxi-Out Time (min)', fontsize=10)
ax.set_title('Flight Distance vs. Air Time (colored by Taxi-Out)', fontweight='bold')
ax.set_xlabel('Distance (miles)')
ax.set_ylabel('Air Time (minutes)')
plt.tight_layout()
plt.savefig('distance_vs_airtime.png', bbox_inches='tight')
plt.show()

## 9. Export Tableau-Ready CSV

In [ ]:
tableau_cols = [
    'year','month','month_name','day_of_month','day_of_week','day_name','fl_date',
    'origin','origin_city_name','origin_state_nm',
    'dep_time','taxi_out','taxi_in','air_time','distance',
    'cancelled','weather_delay','late_aircraft_delay',
    'total_delay','delay_category','primary_cause','is_delayed','high_taxi_out'
]

tableau_df = df[tableau_cols].copy()
tableau_df.to_csv('data/flight_delays_tableau_ready.csv', index=False)
print(f"Exported {len(tableau_df):,} rows → flight_delays_tableau_ready.csv")
print("\nLoad this file into Tableau Public to build the dashboard.")

## 10. Tableau Dashboard Build Guide

**File to load:** `data/flight_delays_tableau_ready.csv`

### Recommended Sheets

| Sheet | Chart Type | Key Fields |
|---|---|---|
| **KPI Header** | Big Number tiles | Total Flights, Cancel Rate, % Delayed, Avg Taxi-Out |
| **Cancellation Rate by Airport** | Horizontal bar | `origin`, `cancelled` |
| **Ground Congestion** | Horizontal bar | `origin`, `taxi_out` avg |
| **Delay Causes** | Donut / Packed bubbles | `primary_cause` |
| **Day of Week Heatmap** | Heatmap | `day_name`, `cancel_rate` |
| **State Map** | Filled map | `origin_state_nm`, `cancel_rate` or `total_flights` |
| **Delay Distribution** | Histogram | `total_delay` |

### Filters to Add
- Month, Day of Week, State, Delay Category, Cancelled (Yes/No)

### Notes
- Use `delay_category` for color-encoding delay severity
- `high_taxi_out` (1/0) works well as a boolean filter for congestion analysis
- State map: use `origin_state_nm` → Tableau will auto-geocode U.S. states
